In [1]:
# Importa las librerías necesarias
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, year, month, round, avg, count
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

# Configura las variables de entorno para la ejecución local de Spark
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["SPARK_HOME"] = r"C:\Users\patri\spark\spark-4.1.1-bin-hadoop3"
os.environ["HADOOP_HOME"] = r"C:\Users\patri\hadoop"

# Configura las variables de entorno para la ejecución local
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["HADOOP_HOME"] = r"C:\Users\patri\hadoop"

# Forza a Windows a buscar el archivo hadoop.dll en la carpeta bin
os.environ["PATH"] = os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

# Configura el formato de visualización de pandas
pd.options.display.float_format = '{:,.2f}'.format

# Inicializa la sesión de Spark
spark = SparkSession.builder \
    .appName("Practica_Big_Data_EBAC") \
    .getOrCreate()

print("Sesión de Spark inicializada.")

Sesión de Spark inicializada.


In [2]:
# Construye la ruta del archivo
ruta_base = r"C:\Users\patri\Downloads"
archivo_csv = "kc_house_data (1).csv"
ruta_completa = os.path.join(ruta_base, archivo_csv)

# Carga el archivo con codificación específica
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "utf-8") \
    .csv(ruta_completa)

# Convierte el identificador principal a texto
df = df.withColumn("id", col("id").cast("string"))

# Prepara el campo fecha_real para Looker Studio
# El formato original viene como '20141013T000000'
df = df.withColumn("fecha_real", to_date(col("date"), "yyyyMMdd'T'HHmmss"))

# Sanitiza caracteres: estandariza nombres de columnas (eliminando guiones o espacios si existieran)
# En este dataset los nombres ya vienen limpios, pero se aplica la técnica por protocolo
df = df.withColumnRenamed("sqft_living", "sqft_living_area")

In [3]:
# Diagnostica la estructura del DataFrame recién cargado
print("Primeras filas (Diagnóstico):")
# Se usa toPandas().head() para respetar el print(df.head()) y el float_format en visualización local
print(df.limit(5).toPandas().head())

print("\nLista de columnas:")
print(df.columns)

Primeras filas (Diagnóstico):
           id             date      price  bedrooms  bathrooms  \
0  7129300520  20141013T000000 221,900.00         3       1.00   
1  6414100192  20141209T000000 538,000.00         3       2.25   
2  5631500400  20150225T000000 180,000.00         2       1.00   
3  2487200875  20141209T000000 604,000.00         4       3.00   
4  1954400510  20150218T000000 510,000.00         3       2.00   

   sqft_living_area  sqft_lot  floors  waterfront  view  ...  sqft_above  \
0              1180      5650    1.00           0     0  ...        1180   
1              2570      7242    2.00           0     0  ...        2170   
2               770     10000    1.00           0     0  ...         770   
3              1960      5000    1.00           0     0  ...        1050   
4              1680      8080    1.00           0     0  ...        1680   

   sqft_basement  yr_built  yr_renovated  zipcode   lat    long  \
0              0      1955             0    98178

In [4]:
# Filtra las propiedades con precio válido y más de cero habitaciones
df_limpio = df.filter((col("price") > 0) & (col("bedrooms") > 0))

# Extrae el año y mes para posibles análisis temporales
df_limpio = df_limpio.withColumn("anio_venta", year(col("fecha_real"))) \
                     .withColumn("mes_venta", month(col("fecha_real")))

# Aplica Window Functions: Clasifica las casas por precio dentro de cada código postal (locación para Looker)
window_spec = Window.partitionBy("zipcode").orderBy(col("price").desc())
df_limpio = df_limpio.withColumn("ranking_precio_zona", rank().over(window_spec))

In [5]:
# Crea el esquema temporal (vista) para consultas
df_limpio.createOrReplaceTempView("vw_casas_base")

# Guarda el SELECT en una nueva VIEW analítica
spark.sql("""
    CREATE OR REPLACE TEMP VIEW vw_analisis_habitaciones AS
    SELECT 
        bedrooms,
        COUNT(id) AS total_propiedades,
        ROUND(AVG(price), 2) AS precio_promedio_usd
    FROM vw_casas_base
    GROUP BY bedrooms
    ORDER BY bedrooms ASC
""")

# Realiza la comprobación obligatoria de la vista (Equivalente en Spark a SELECT TOP(3))
print("Comprobación de la vista analítica (TOP 3):")
spark.sql("SELECT * FROM vw_analisis_habitaciones LIMIT 3").show()

Comprobación de la vista analítica (TOP 3):
+--------+-----------------+-------------------+
|bedrooms|total_propiedades|precio_promedio_usd|
+--------+-----------------+-------------------+
|       1|              199|          317642.88|
|       2|             2760|          401372.68|
|       3|             9824|          466232.08|
+--------+-----------------+-------------------+



In [6]:
# Define la ruta de salida, asegurando compatibilidad UTF-8 y particionamiento
ruta_exportacion = os.path.join(ruta_base, "kc_house_export_looker")

# Exporta los datos en formato CSV particionado por condición, ideal para consumo o respaldo
df_limpio.write \
    .mode("overwrite") \
    .partitionBy("condition") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .csv(ruta_exportacion)

print(f"Datos particionados y exportados exitosamente en: {ruta_exportacion}")

Datos particionados y exportados exitosamente en: C:\Users\patri\Downloads\kc_house_export_looker
